# Module 6 Homework Solutions

This notebook solves the exercises from `homework.md` using the local files in `cohorts/2026/06-batch/`.

Selected answers:
- Q1: `spark.version == 4.1.1`
- Q2: `25 MB`
- Q3: `162,604`
- Q4: `90.6467 hours`
- Q5: `4040`
- Q6: `Arden Heights`


In [ ]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

candidate_dirs = [Path.cwd(), Path.cwd() / 'cohorts/2026/06-batch']
BASE_DIR = next((path.resolve() for path in candidate_dirs if (path / 'yellow_tripdata_2025-11.parquet').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Could not locate yellow_tripdata_2025-11.parquet')

YELLOW_PATH = BASE_DIR / 'yellow_tripdata_2025-11.parquet'
ZONES_PATH = BASE_DIR / 'taxi_zone_lookup.csv'
Q2_OUTPUT_PATH = BASE_DIR / 'tmp_hw6_q2_parquet'

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('de-zoomcamp-2026-hw6')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

df = spark.read.parquet(str(YELLOW_PATH))
BASE_DIR


## Question 1

Create a local Spark session and inspect `spark.version`.


In [ ]:
pip install pyspark==4.1.1
spark.version


Answer: `4.1.1`


## Question 2

Repartition the Yellow November 2025 dataframe into 4 partitions, save it to parquet, and compute the average parquet file size in MB.


In [ ]:
if Q2_OUTPUT_PATH.exists():
    shutil.rmtree(Q2_OUTPUT_PATH)

df_q2 = df.repartition(4)
df_q2.write.mode('overwrite').parquet(str(Q2_OUTPUT_PATH))

parquet_files = sorted(Q2_OUTPUT_PATH.glob('*.parquet'))
file_sizes_mb = [path.stat().st_size / 1_000_000 for path in parquet_files]
avg_file_size_mb = sum(file_sizes_mb) / len(file_sizes_mb)

avg_file_size_mb


Answer: `25 MB` (computed average: `25.59857325 MB`)


## Question 3

Count trips that started on `2025-11-15`.


In [ ]:
df.filter(F.to_date('tpep_pickup_datetime') == F.lit('2025-11-15')).count()


Answer: `162,604`


## Question 4

Compute the maximum trip duration in hours.


In [ ]:
df.select(
    ((F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600.0).alias('duration_hours')
).agg(F.max('duration_hours')).first()[0]


Answer: `90.64666666666666` hours, which matches the `90.6` option.


## Question 5

Spark UI runs on the default local port `4040`.

Answer: `4040`


## Question 6

Join the trip data with the taxi zone lookup table and find the least frequent pickup zone.


In [ ]:
zones = spark.read.option('header', True).csv(str(ZONES_PATH))
zones = zones.withColumn('LocationID_int', F.col('LocationID').cast('int'))

least_frequent_pickups = (
    df.groupBy('PULocationID').count().alias('pickup_counts')
    .join(zones.alias('zones'), F.col('pickup_counts.PULocationID') == F.col('zones.LocationID_int'), 'left')
    .select(
        F.col('pickup_counts.PULocationID').alias('PULocationID'),
        F.col('zones.Zone').alias('Zone'),
        F.col('count')
    )
    .orderBy(F.col('count').asc(), F.col('PULocationID').asc())
)

least_frequent_pickups.show(10, truncate=False)


Answer: `Arden Heights` is a valid choice. There is a tie at `count = 1`, and `Governor's Island/Ellis Island/Liberty Island` is also valid per the homework note.
